In [2]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train_full = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_train_full_clean.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_test_future.parquet')

print(train_full.shape, test.shape)

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

(749609, 34) (114111, 34)


In [3]:
# ============================================================
# 0. 라이브러리 불러오기
# ============================================================

import numpy as np

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

In [4]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [6]:
# ============================================================
# 2. 강우사상별 event_id 부여, 컬럼 생성
# ============================================================

# 1-1. 전체 timestamp 추출(중복없이), 시간순 정렬
event_time = (
    train_full[['timestamp']] 
    .drop_duplicates() 
    .sort_values('timestamp') 
    .reset_index(drop = True) 
)

# 1-2. diff 계산
event_time['event_time_diff'] = (
    event_time['timestamp']
    .diff()
)

# 1-3. 차분값이 1시간이 아닌 경우를 event 경계로 추출
cond_split_time = (
    event_time['event_time_diff'] != pd.Timedelta(hours=1)
)

# 1-4. 강우사상 시간대별 event_id 부여
event_time['event_id'] = cond_split_time.cumsum()

# 시험처가 제시한 강우사상 개수와 맞는지 확인: 122개
print(event_time['event_id'].nunique())

# 기존 셋에 합치기

train_full['event_id'] = (
    train_full['timestamp'].map(
        event_time.set_index('timestamp')['event_id']
    )
)

print('강우사상 개수:', train_full['event_id'].nunique())
print('event_id 결측:', train_full['event_id'].isna().sum())
print('데이터 크기:', train_full.shape)

122
강우사상 개수: 122
event_id 결측: 0
데이터 크기: (749609, 35)


In [7]:
# ============================================================
# 3. 시간사상 Group K-Fold 설정 및 Fold 배정 저장
# ============================================================

from sklearn.model_selection import GroupKFold

# ------------------------------------------------------------
# 3-1. Group 설정
# 같은 시간사상에 속한 관측치는 반드시 같은 Fold에 들어가도록 설정
# ------------------------------------------------------------

kf_groups = train_full['event_id']

print(f'<시간사상 Group 확인>')
print(f'전체 학습 행 수   : {len(train_full):,}')
print(f'시간사상 개수     : {kf_groups.nunique():,}')
print()

assert kf_groups.notnull().all(), 'event_id 결측치 존재'


# ------------------------------------------------------------
# 3-2. GroupKFold 정의
# ------------------------------------------------------------

N_SPLITS = 5

gkf = GroupKFold(n_splits=N_SPLITS)

# 시간사상별 Fold 번호 저장용 데이터프레임
kf_fold_f = pd.DataFrame({
    'event_id': kf_groups.unique()  # event_id 하나당 한 셀로 생긴 df
})

kf_fold_f['fold_num'] = 0  # 일단 'fold_num 컬럼 0으로 다 채워놓고


# ------------------------------------------------------------
# 3-3. Fold 배정 및 검증
# ------------------------------------------------------------

for fold, (train_idx, valid_idx) in enumerate(  # 반환값은 train과 val의 인덱스 셋
    gkf.split(train_full, groups=kf_groups),  # 분할 대상, 분할때 지킬 그룹기준
    start=1
):
    train_groups = set(kf_groups.iloc[train_idx])  # 집합으로 만들어서
    valid_groups = set(kf_groups.iloc[valid_idx])

    overlap = train_groups & valid_groups  # 겹치는겁 없는지 교집합으로 확인

    # 학습·검증 사상 중복 여부 먼저 확인
    assert len(overlap) == 0, f'Fold {fold}: 시간사상 중복'

    # 검증 Fold에 해당하는 시간사상에 Fold 번호 부여
    kf_fold_f.loc[
        kf_fold_f['event_id'].isin(valid_groups),
        'fold_num'
    ] = fold  # 맞는 인덱스에 가서 번호 붙이기
              # 이후에 순회하면서 번호를 붙여나감

    print(
        f'Fold {fold} | '
        f'train={len(train_idx):,} | '
        f'valid={len(valid_idx):,} | '
        f'train events={len(train_groups):,} | '
        f'valid events={len(valid_groups):,} | '
        f'event overlap={len(overlap)}'
    )


# ------------------------------------------------------------
# 3-4. Fold 배정 결과 최종 검증
# ------------------------------------------------------------

assert kf_fold_f['event_id'].is_unique, 'event_id 중복'
assert kf_fold_f['fold_num'].between(1, N_SPLITS).all(), 'Fold 배정 누락'

assert set(kf_fold_f['event_id']) == set(kf_groups), '시간사상 누락'

# event_id 기준 정렬
kf_fold_f = kf_fold_f.sort_values('event_id').reset_index(drop=True)  # 정렬 후 인덱스 0부터 다시부여

print('\n<Fold별 시간사상 개수>')
print(kf_fold_f['fold_num'].value_counts().sort_index())

display(kf_fold_f)


# ------------------------------------------------------------
# 3-5. Fold 배정 CSV 저장
# ------------------------------------------------------------

kf_fold_f.to_csv(
    PROJECT_ROOT / 'config' / '0_kf_train_full_match.csv',
    index=False
)

print('\nFold 배정 CSV 저장 완료')


# ------------------------------------------------------------
# 3-6. train_full에 Fold 번호 부여
# ------------------------------------------------------------

train_full['fold_num'] = train_full['event_id'].map(
    kf_fold_f.set_index('event_id')['fold_num']
)

# Fold 배정 검증
assert train_full['fold_num'].notna().all(), 'Fold 배정 누락'

print(train_full['fold_num'].value_counts().sort_index())

<시간사상 Group 확인>
전체 학습 행 수   : 749,609
시간사상 개수     : 122

Fold 1 | train=599,625 | valid=149,984 | train events=100 | valid events=22 | event overlap=0
Fold 2 | train=599,624 | valid=149,985 | train events=97 | valid events=25 | event overlap=0
Fold 3 | train=599,673 | valid=149,936 | train events=97 | valid events=25 | event overlap=0
Fold 4 | train=599,830 | valid=149,779 | train events=97 | valid events=25 | event overlap=0
Fold 5 | train=599,684 | valid=149,925 | train events=97 | valid events=25 | event overlap=0

<Fold별 시간사상 개수>
fold_num
1    22
2    25
3    25
4    25
5    25
Name: count, dtype: int64


,event_id,fold_num
0,1,1
1,2,2
2,3,3
3,4,4
4,5,5
...,...,...
117,118,4
118,119,1
119,120,1
120,121,5



Fold 배정 CSV 저장 완료
fold_num
1    149984
2    149985
3    149936
4    149779
5    149925
Name: count, dtype: int64


In [8]:
# ============================================================
# 4. Tidal / Normal 데이터 분리
# ============================================================

# 조석 영향 의심 관측소 9개 (기존과 동일)
tidal_stations = [
    'ST0110', 'ST0326', 'ST0159',
    'ST0358', 'ST0347', 'ST0415',
    'ST0292', 'ST0313', 'ST0234'
]

# 4-1. Train 분리
tidal_train = train_full.loc[
    train_full['station_id'].isin(tidal_stations)
].copy()

normal_train = train_full.loc[
    ~train_full['station_id'].isin(tidal_stations)
].copy()

# 4-2. Test 분리
tidal_test = test.loc[
    test['station_id'].isin(tidal_stations)
].copy()

normal_test = test.loc[
    ~test['station_id'].isin(tidal_stations)
].copy()

# 4-3. 분리 결과 확인
print('<Train 분리 결과>')
print(f'Train 전체 : {len(train_full):,}')
print(f'Tidal      : {len(tidal_train):,}')
print(f'Normal     : {len(normal_train):,}')

print('\n<Test 분리 결과>')
print(f'Test 전체  : {len(test):,}')
print(f'Tidal      : {len(tidal_test):,}')
print(f'Normal     : {len(normal_test):,}')

# 4-4. 데이터 분리 검증
assert len(tidal_train) + len(normal_train) == len(train_full)
assert len(tidal_test) + len(normal_test) == len(test)

print('\n데이터 분리 검증 완료')

<Train 분리 결과>
Train 전체 : 749,609
Tidal      : 14,251
Normal     : 735,358

<Test 분리 결과>
Test 전체  : 114,111
Tidal      : 2,169
Normal     : 111,942

데이터 분리 검증 완료


In [13]:
# ============================================================
# 6. 독립변수 / 종속변수 분리
# ============================================================

# 6-1. Tidal
X_tidal_train = tidal_train[num_cols_fe + ['station_id']].copy()
X_tidal_test = tidal_test[num_cols_fe + ['station_id']].copy()

y_tidal_train = tidal_train[target_col]
y_tidal_test = tidal_test[target_col]

# 6-2. Normal
X_normal_train = normal_train[num_cols_fe + ['station_id']].copy()
X_normal_test = normal_test[num_cols_fe + ['station_id']].copy()

y_normal_train = normal_train[target_col]
y_normal_test = normal_test[target_col]

# 6-3. CatBoost 범주형 변수 처리
for X in [
    X_tidal_train, X_tidal_test,
    X_normal_train, X_normal_test
]:
    X['station_id'] = X['station_id'].astype(str)

# 6-4. Shape 확인
# 종속변수, 'event_id', 'time_stamp', 'fold_num'이 빠짐
# 'fold_num'은 나중에 모델에 넣을때 때고 넣어야함(불필요한 학습)
# 그래서 인덱스로 'fold_num'을 매칭해서 쓸 용도로 tidal_train을 쓰게될 것
print('<Tidal>')
print('X_train:', X_tidal_train.shape)
print('X_test :', X_tidal_test.shape)
print('y_train:', y_tidal_train.shape)
print('y_test :', y_tidal_test.shape)

print('\n<Normal>')
print('X_train:', X_normal_train.shape)
print('X_test :', X_normal_test.shape)
print('y_train:', y_normal_train.shape)
print('y_test :', y_normal_test.shape)

<Tidal>
X_train: (14251, 32)
X_test : (2169, 32)
y_train: (14251,)
y_test : (2169,)

<Normal>
X_train: (735358, 32)
X_test : (111942, 32)
y_train: (735358,)
y_test : (111942,)


In [12]:
print('모델 입력에서 제외된 컬럼:')
print(train_full.columns.difference(X_tidal_train.columns).tolist())

모델 입력에서 제외된 컬럼:
['event_id', 'fold_num', 'target_maxrise_6h', 'timestamp']


In [14]:
# ============================================================
# 7. Tidal CatBoost 5-Fold 학습 및 평가
# ============================================================

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error
import numpy as np
import pandas as pd

# 7-1. 결과 저장용 변수
tidal_oof_pred = pd.Series(  # tidal_oof용 예측값은 fold별 예측값을 직렬로 이어 붙이므로 시리즈
    index=X_tidal_train.index,
    dtype=float
)

tidal_test_preds = pd.DataFrame(  # test 예측값은 fold별 예측값을 병렬로 놓고 평균, df로 정의
    index=X_tidal_test.index
)

tidal_fold_results = []

# 7-2. Fold별 학습
for fold in range(1, N_SPLITS + 1):

    print(f'\n========== Tidal Fold {fold} ==========')

    # 기존 Fold 번호로 분할
    train_mask = tidal_train['fold_num'] != fold  # fold 숫자 아닌게 train
    valid_mask = tidal_train['fold_num'] == fold  # fold 숫자에 딱 맞는게 val
    # tidal_train에서 fold_num으로 조건을 설정해서

    X_tr = X_tidal_train.loc[train_mask]  # 집어넣기
    X_val = X_tidal_train.loc[valid_mask]

    y_tr = y_tidal_train.loc[train_mask]
    y_val = y_tidal_train.loc[valid_mask]

    assert len(X_tr) > 0 and len(X_val) > 0, f'Fold {fold}: 학습/검증 데이터 없음'

    # CatBoost 최종 선정모델
    model = CatBoostRegressor(
        iterations=3000,
        loss_function='RMSE',
        random_seed=42,
        thread_count=-1
    )

    model.fit(
        X_tr,
        y_tr,
        cat_features=['station_id'],
        eval_set=(X_val, y_val),
        early_stopping_rounds=100,
        verbose=200
    )

    # 7-3. OOF 예측 및 Fold RMSE
    val_pred = model.predict(X_val)  # fold안에서 val 예측은 OOF용으로 들어가고

    tidal_oof_pred.loc[X_val.index] = val_pred

    fold_oof_rmse = root_mean_squared_error(y_val, val_pred)

    # 7-4. Test 전체 예측 및 Fold RMSE
    test_pred = model.predict(X_tidal_test)  # test 전체 예측

    tidal_test_preds[f'fold_{fold}'] = test_pred

    fold_test_rmse = root_mean_squared_error(y_tidal_test, test_pred)

    # 7-5. Fold 결과 저장
    tidal_fold_results.append({
        'fold': fold,
        'train_count': len(X_tr),
        'valid_count': len(X_val),
        'best_iteration': model.get_best_iteration(),
        'oof_rmse': fold_oof_rmse,
        'test_rmse': fold_test_rmse
    })

    print(
        f'Fold {fold} | '
        f'OOF RMSE={fold_oof_rmse:.6f} | '
        f'Test RMSE={fold_test_rmse:.6f} | '
        f'Best Iter={model.get_best_iteration()}'
    )


# ============================================================
# 7-6. 전체 OOF 및 Test 앙상블 평가
# ============================================================

# OOF 예측 누락 확인
assert not tidal_oof_pred.isnull().any(), 'Tidal OOF 예측 누락'

# 전체 Tidal OOF RMSE
tidal_oof_rmse = root_mean_squared_error(y_tidal_train, tidal_oof_pred)

# 5개 Fold Test 예측 평균
tidal_test_ensemble = tidal_test_preds.mean(axis=1)

# 전체 Tidal Test RMSE
tidal_test_rmse = root_mean_squared_error(y_tidal_test, tidal_test_ensemble)

# Fold별 결과표
tidal_fold_results = pd.DataFrame(tidal_fold_results)

print('\n<Tidal Fold별 학습 결과>')
display(tidal_fold_results)

print('\n<Tidal 최종 성능>')
print(f'전체 OOF RMSE        : {tidal_oof_rmse:.6f}')
print(f'Test 앙상블 RMSE     : {tidal_test_rmse:.6f}')


========== Tidal Fold 1 ==========
Learning rate set to 0.038166
0:	learn: 1.6968054	test: 1.7675073	best: 1.7675073 (0)	total: 143ms	remaining: 7m 7s
200:	learn: 0.3947512	test: 0.5023623	best: 0.5023623 (200)	total: 5.31s	remaining: 1m 13s
400:	learn: 0.3355905	test: 0.4708035	best: 0.4708035 (400)	total: 10.7s	remaining: 1m 9s
600:	learn: 0.3014610	test: 0.4578791	best: 0.4576996 (598)	total: 16.4s	remaining: 1m 5s
800:	learn: 0.2769834	test: 0.4482963	best: 0.4482227 (799)	total: 22.3s	remaining: 1m 1s
1000:	learn: 0.2590853	test: 0.4435206	best: 0.4435206 (1000)	total: 28.4s	remaining: 56.7s
1200:	learn: 0.2426534	test: 0.4424713	best: 0.4423458 (1191)	total: 34.2s	remaining: 51.3s
1400:	learn: 0.2292017	test: 0.4407490	best: 0.4407184 (1399)	total: 40.2s	remaining: 45.9s
1600:	learn: 0.2174681	test: 0.4398281	best: 0.4398281 (1600)	total: 46.1s	remaining: 40.3s
1800:	learn: 0.2074587	test: 0.4387945	best: 0.4385728 (1764)	total: 52.3s	remaining: 34.8s
2000:	learn: 0.1982086	test

,fold,train_count,valid_count,best_iteration,oof_rmse,test_rmse
0,1,11381,2870,1979,0.438215,0.359602
1,2,11421,2830,2999,0.363745,0.365767
2,3,11377,2874,2090,0.334071,0.366590
3,4,11427,2824,2433,0.339350,0.374119
4,5,11398,2853,2386,0.350624,0.356292



<Tidal 최종 성능>
전체 OOF RMSE        : 0.367269
Test 앙상블 RMSE     : 0.355481


In [15]:
# ============================================================
# 8. Normal CatBoost 5-Fold 학습 및 평가
# ============================================================

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error
import numpy as np
import pandas as pd

# 8-1. 결과 저장용 변수
normal_oof_pred = pd.Series(
    index=X_normal_train.index,
    dtype=float
)

normal_test_preds = pd.DataFrame(
    index=X_normal_test.index
)

normal_fold_results = []

# 8-2. Fold별 학습
for fold in range(1, N_SPLITS + 1):

    print(f'\n========== Normal Fold {fold} ==========')

    # 기존 Fold 번호로 분할
    train_mask = normal_train['fold_num'] != fold
    valid_mask = normal_train['fold_num'] == fold

    X_tr = X_normal_train.loc[train_mask]
    X_val = X_normal_train.loc[valid_mask]

    y_tr = y_normal_train.loc[train_mask]
    y_val = y_normal_train.loc[valid_mask]

    assert len(X_tr) > 0 and len(X_val) > 0, f'Fold {fold}: 학습/검증 데이터 없음'

    # CatBoost 최종 선정모델
    model = CatBoostRegressor(
        iterations=3000,
        loss_function='RMSE',
        random_seed=42,
        thread_count=-1
    )

    model.fit(
        X_tr,
        y_tr,
        cat_features=['station_id'],
        eval_set=(X_val, y_val),
        early_stopping_rounds=100,
        verbose=200
    )

    # 8-3. OOF 예측 및 Fold RMSE
    val_pred = model.predict(X_val)

    normal_oof_pred.loc[X_val.index] = val_pred

    fold_oof_rmse = root_mean_squared_error(y_val, val_pred)

    # 8-4. Test 전체 예측 및 Fold RMSE
    test_pred = model.predict(X_normal_test)

    normal_test_preds[f'fold_{fold}'] = test_pred

    fold_test_rmse = root_mean_squared_error(y_normal_test, test_pred)

    # 8-5. Fold 결과 저장
    normal_fold_results.append({
        'fold': fold,
        'train_count': len(X_tr),
        'valid_count': len(X_val),
        'best_iteration': model.get_best_iteration(),
        'oof_rmse': fold_oof_rmse,
        'test_rmse': fold_test_rmse
    })

    print(
        f'Fold {fold} | '
        f'OOF RMSE={fold_oof_rmse:.6f} | '
        f'Test RMSE={fold_test_rmse:.6f} | '
        f'Best Iter={model.get_best_iteration()}'
    )


# ============================================================
# 8-6. 전체 OOF 및 Test 앙상블 평가
# ============================================================

# OOF 예측 누락 확인
assert not normal_oof_pred.isnull().any(), 'Normal OOF 예측 누락'

# 전체 Normal OOF RMSE
normal_oof_rmse = root_mean_squared_error(y_normal_train, normal_oof_pred)

# 5개 Fold Test 예측 평균
normal_test_ensemble = normal_test_preds.mean(axis=1)

# 전체 Normal Test RMSE
normal_test_rmse = root_mean_squared_error(y_normal_test, normal_test_ensemble)

# Fold별 결과표
normal_fold_results = pd.DataFrame(normal_fold_results)

print('\n<Normal Fold별 학습 결과>')
display(normal_fold_results)

print('\n<Normal 최종 성능>')
print(f'전체 OOF RMSE        : {normal_oof_rmse:.6f}')
print(f'Test 앙상블 RMSE     : {normal_test_rmse:.6f}') 


========== Normal Fold 1 ==========
Learning rate set to 0.070904
0:	learn: 0.2732156	test: 0.2821174	best: 0.2821174 (0)	total: 110ms	remaining: 5m 29s
200:	learn: 0.2090740	test: 0.2346370	best: 0.2346370 (200)	total: 18.5s	remaining: 4m 17s
400:	learn: 0.1995046	test: 0.2316771	best: 0.2316335 (395)	total: 36.4s	remaining: 3m 56s
600:	learn: 0.1930595	test: 0.2314874	best: 0.2312638 (541)	total: 54.3s	remaining: 3m 36s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.2312638212
bestIteration = 541

Shrink model to first 542 iterations.
Fold 1 | OOF RMSE=0.231264 | Test RMSE=0.205415 | Best Iter=541

========== Normal Fold 2 ==========
Learning rate set to 0.070903
0:	learn: 0.2905433	test: 0.2001787	best: 0.2001787 (0)	total: 148ms	remaining: 7m 22s
200:	learn: 0.2233578	test: 0.1692181	best: 0.1692181 (200)	total: 34.1s	remaining: 7m 54s
400:	learn: 0.2129015	test: 0.1664374	best: 0.1664356 (398)	total: 59.9s	remaining: 6m 28s
600:	learn: 0.2058701	test: 0.1655

,fold,train_count,valid_count,best_iteration,oof_rmse,test_rmse
0,1,588244,147114,541,0.231264,0.205415
1,2,588203,147155,984,0.165122,0.202403
2,3,588296,147062,2276,0.259769,0.198372
3,4,588403,146955,1100,0.209466,0.201997
4,5,588286,147072,1106,0.207124,0.201146



<Normal 최종 성능>
전체 OOF RMSE        : 0.216793
Test 앙상블 RMSE     : 0.199124


In [19]:
# ============================================================
# 9. Tidal + Normal 통합 OOF 및 Test RMSE 평가
# ============================================================

from sklearn.metrics import root_mean_squared_error

# 9-1. 전체 Train OOF 예측값 통합
full_oof_pred = pd.concat([
    tidal_oof_pred,
    normal_oof_pred
])

# 9-2. 전체 Test 앙상블 예측값 통합
full_test_pred = pd.concat([
    tidal_test_ensemble,
    normal_test_ensemble
])

# 9-3. 통합 예측값 검증
assert full_oof_pred.index.is_unique, 'OOF 인덱스 중복'
assert full_test_pred.index.is_unique, 'Test 인덱스 중복'

assert set(full_oof_pred.index) == set(train_full.index), 'Train 인덱스 불일치'
assert set(full_test_pred.index) == set(test.index), 'Test 인덱스 불일치'

assert not full_oof_pred.isnull().any(), '전체 OOF 예측 누락'
assert not full_test_pred.isnull().any(), '전체 Test 예측 누락'

# 9-4. 원본 인덱스 순서로 예측값 정렬
full_oof_pred = full_oof_pred.reindex(train_full.index)
full_test_pred = full_test_pred.reindex(test.index)

print('전체 OOF 예측 행 수 :', len(full_oof_pred))
print('전체 Test 예측 행 수:', len(full_test_pred))

# 9-5. 전체 RMSE 계산
full_oof_rmse = root_mean_squared_error(
    train_full[target_col],
    full_oof_pred
)

full_test_rmse = root_mean_squared_error(
    test[target_col],
    full_test_pred
)

# 9-6. 최종 성능 비교표
final_results = pd.DataFrame({
    'OOF_RMSE': [
        tidal_oof_rmse,
        normal_oof_rmse,
        full_oof_rmse
    ],
    'Test_RMSE': [
        tidal_test_rmse,
        normal_test_rmse,
        full_test_rmse
    ]
}, index=['Tidal', 'Normal', '전체'])

print('\n<최종 모델 성능 비교>')
display(final_results)

전체 OOF 예측 행 수 : 749609
전체 Test 예측 행 수: 114111

<최종 모델 성능 비교>


,OOF_RMSE,Test_RMSE
Tidal,0.367269,0.355481
Normal,0.216793,0.199124
전체,0.220613,0.203221


In [17]:
print('인덱스 순서 일치:', full_oof_pred.index.equals(train_full.index))
print('인덱스 구성 일치:', set(full_oof_pred.index) == set(train_full.index))

인덱스 순서 일치: False
인덱스 구성 일치: True


In [ ]:
# 모델 구조 선정: 단일 CatBoost와 Tidal/Normal 분리모델의 K-Fold OOF RMSE를 비교한다. 
# Test 결과도 참고하되, K-Fold에서 더 안정적이고 우수한 구조를 우선한다.


# Early Stopping 적용: K-Fold 또는 기존 단일 홀드아웃 Validation에서 확인한 best_iteration_을 활용해 최종 반복 횟수를 결정한다.
# Tidal과 Normal 각각의 최적 반복 횟수를 별도로 확정